# Tutorial: Frequency Analysis AC Sweep Spec Walkthrough

**Audience:** backend/frontend contributors integrating the new AC sweep contract.

**Prerequisites:**

- Pulsim Python bindings available (`PYTHONPATH=build/python`),
- familiarity with YAML netlists and `SimulationOptions`,
- basic Bode interpretation (magnitude/phase).

**Learning goals:**

- parse and verify `simulation.frequency_analysis` from YAML,
- run AC sweep through class and procedural APIs,
- inspect typed diagnostics (`FrequencyAnalysisError`) and reason codes,
- validate anchor policy (`dc`, `periodic`, `averaged`, `auto` fallback),
- validate benchmark/KPI artifacts for AC coverage.

## Outline

1. Setup and imports.
2. Load AC RC low-pass YAML and inspect mapped options.
3. Run class API and inspect response arrays + metadata.
4. Confirm procedural API parity.
5. Exercise anchor-selection behavior (`auto` periodic and fallback-to-averaged).
6. Exercise structured AC failures with `raise_on_failure=True`.
7. Run AC benchmark scenarios and AC KPI gate.

In [1]:
# Setup cell: deterministic and self-contained
from __future__ import annotations

import json
import math
import os
import sys
import tempfile
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'benchmarks').is_dir():
            return candidate
    raise RuntimeError('Could not locate repository root')


repo_root = find_repo_root(Path.cwd())
if Path.cwd().resolve() != repo_root:
    os.chdir(repo_root)

build_python = repo_root / 'build' / 'python'
if build_python.is_dir() and str(build_python) not in sys.path:
    sys.path.insert(0, str(build_python))

benchmarks_dir = repo_root / 'benchmarks'
if str(benchmarks_dir) not in sys.path:
    sys.path.insert(0, str(benchmarks_dir))

try:
    import pulsim as ps
except Exception as exc:
    raise RuntimeError(
        'Failed to import pulsim. Build bindings and run with PYTHONPATH=build/python.'
    ) from exc

import benchmark_runner
import freeze_kpi_baseline
import kpi_gate

print('repo_root:', repo_root)
print('pulsim version:', getattr(ps, '__version__', 'unknown'))
print('build/python injected:', str(build_python) in sys.path)

RuntimeError: Failed to import pulsim. Build bindings and run with PYTHONPATH=build/python.

## Step 1 - YAML Contract Mapping (`simulation.frequency_analysis`)

This loads the canonical AC example (`ac_rc_lowpass.yaml`) and verifies that parser mapping preserves mode, anchor, sweep, and ports.

In [2]:
parser_opts = ps.YamlParserOptions()
parser_opts.strict = False
parser = ps.YamlParser(parser_opts)

netlist_path = repo_root / 'benchmarks' / 'circuits' / 'ac_rc_lowpass.yaml'
circuit, sim_options = parser.load(str(netlist_path))
if parser.errors:
    raise RuntimeError('YAML parser errors\n' + '\n'.join(parser.errors))
fa = sim_options.frequency_analysis
snapshot = {
    'enabled': bool(fa.enabled),
    'mode': fa.mode.name,
    'anchor_mode': fa.anchor_mode.name,
    'sweep_scale': fa.sweep_scale.name,
    'f_start_hz': float(fa.f_start_hz),
    'f_stop_hz': float(fa.f_stop_hz),
    'points': int(fa.points),
    'injection_current_amplitude': float(fa.injection_current_amplitude),
    'perturbation_port': {
        'positive': fa.perturbation_port.positive_node,
        'negative': fa.perturbation_port.negative_node,
    },
    'output_port': {
        'positive': fa.output_port.positive_node,
        'negative': fa.output_port.negative_node,
    },
}
print(json.dumps(snapshot, indent=2))

assert fa.enabled is True
assert fa.mode == ps.FrequencyAnalysisMode.OpenLoopTransfer
assert fa.anchor_mode == ps.FrequencyAnchorMode.DC
assert fa.sweep_scale == ps.FrequencySweepScale.Logarithmic
assert fa.points == 80
assert abs(fa.f_start_hz - 10.0) < 1e-12
assert abs(fa.f_stop_hz - 100000.0) < 1e-12
assert fa.perturbation_port.positive_node == 'in'
assert fa.output_port.positive_node == 'out'

{
  "enabled": true,
  "mode": "OpenLoopTransfer",
  "anchor_mode": "DC",
  "sweep_scale": "Logarithmic",
  "f_start_hz": 10.0,
  "f_stop_hz": 100000.0,
  "points": 80,
  "injection_current_amplitude": 1.0,
  "perturbation_port": {
    "positive": "in",
    "negative": "0"
  },
  "output_port": {
    "positive": "out",
    "negative": "0"
  }
}


## Step 2 - Class API AC Sweep + Metadata + Undefined Reasons

Run `Simulator.run_frequency_analysis(...)` and validate physics-facing expectations for the RC low-pass.

In [3]:
sim = ps.Simulator(circuit, sim_options)
ac_class = sim.run_frequency_analysis(sim_options.frequency_analysis)

print('success:', ac_class.success)
print('diagnostic:', ac_class.diagnostic.name)
print('anchor_mode_selected:', ac_class.anchor_mode_selected.name)
print('message:', ac_class.message)
print('points:', len(ac_class.frequency_hz))
print('first/last magnitude_db:', ac_class.magnitude_db[0], ac_class.magnitude_db[-1])
print('first/last phase_deg:', ac_class.phase_deg[0], ac_class.phase_deg[-1])

assert ac_class.success is True
assert len(ac_class.frequency_hz) == sim_options.frequency_analysis.points
assert len(ac_class.magnitude_db) == len(ac_class.frequency_hz)
assert len(ac_class.phase_deg) == len(ac_class.frequency_hz)

# RC low-pass qualitative checks.
assert ac_class.magnitude_db[0] > -0.5
assert ac_class.magnitude_db[-1] < -35.0
assert ac_class.phase_deg[-1] < -60.0

f_cutoff = 1.0 / (2.0 * math.pi * 1_000.0 * 1e-6)
idx_fc = min(range(len(ac_class.frequency_hz)), key=lambda i: abs(ac_class.frequency_hz[i] - f_cutoff))
print('f_c(ideal)=', f_cutoff, 'Hz  |  measured magnitude_db at nearest point=', ac_class.magnitude_db[idx_fc])
assert abs(ac_class.magnitude_db[idx_fc] + 3.0) < 2.0

meta = ac_class.channel_metadata
print('metadata keys:', sorted(meta.keys()))
print('frequency_hz unit/domain:', meta['frequency_hz'].unit, meta['frequency_hz'].domain)
print('magnitude_db unit/domain:', meta['magnitude_db'].unit, meta['magnitude_db'].domain)
print('phase_deg unit/domain:', meta['phase_deg'].unit, meta['phase_deg'].domain)

assert meta['frequency_hz'].unit == 'Hz'
assert meta['magnitude_db'].unit == 'dB'
assert meta['phase_deg'].unit == 'deg'
assert meta['magnitude_db'].domain == 'frequency'

print('gain_crossover_reason:', ac_class.gain_crossover_reason.name)
print('phase_margin_reason:', ac_class.phase_margin_reason.name)
print('phase_crossover_reason:', ac_class.phase_crossover_reason.name)
print('gain_margin_reason:', ac_class.gain_margin_reason.name)

# Passive RC in this range has undefined margins for explicit reasons.
assert ac_class.gain_crossover_reason == ps.FrequencyMetricUndefinedReason.NoGainCrossover
assert ac_class.phase_margin_reason == ps.FrequencyMetricUndefinedReason.NoGainCrossover
assert ac_class.phase_crossover_reason == ps.FrequencyMetricUndefinedReason.NoPhaseCrossover
assert ac_class.gain_margin_reason == ps.FrequencyMetricUndefinedReason.NoPhaseCrossover

success: True
diagnostic: None
anchor_mode_selected: DC
message: Frequency analysis completed (anchor=dc)
points: 80
first/last magnitude_db: -0.017111504344580936 -55.963608367956105
first/last phase_deg: -3.5952737798681746 -89.90881101171665
f_c(ideal)= 159.15494309189535 Hz  |  measured magnitude_db at nearest point= -3.1460549847955717
metadata keys: ['frequency_hz', 'magnitude', 'magnitude_db', 'phase_deg', 'response_imag', 'response_real']
frequency_hz unit/domain: Hz frequency
magnitude_db unit/domain: dB frequency
phase_deg unit/domain: deg frequency
gain_crossover_reason: NoGainCrossover
phase_margin_reason: NoGainCrossover
phase_crossover_reason: NoPhaseCrossover
gain_margin_reason: NoPhaseCrossover


## Step 3 - Procedural API Parity

The procedural entrypoint must return the same sampled response as the class entrypoint for equivalent inputs.

In [4]:
ac_proc = ps.run_frequency_analysis(circuit, sim_options.frequency_analysis)


def max_abs_diff(a, b):
    return max((abs(x - y) for x, y in zip(a, b)), default=0.0)

mag_diff = max_abs_diff(ac_class.magnitude_db, ac_proc.magnitude_db)
phase_diff = max_abs_diff(ac_class.phase_deg, ac_proc.phase_deg)
freq_diff = max_abs_diff(ac_class.frequency_hz, ac_proc.frequency_hz)

print('max |delta frequency_hz| =', freq_diff)
print('max |delta magnitude_db| =', mag_diff)
print('max |delta phase_deg| =', phase_diff)

assert ac_proc.success is True
assert freq_diff < 1e-12
assert mag_diff < 1e-12
assert phase_diff < 1e-12

max |delta frequency_hz| = 0.0
max |delta magnitude_db| = 0.0
max |delta phase_deg| = 0.0


## Step 4 - Anchor Policy Behavior (`auto`)

This cell reproduces two deterministic contract cases:

- `auto` picks `periodic` when periodic excitation is available.
- `auto` falls back from periodic pre-anchor to `averaged` when periodic solve cannot converge under constrained settings.

In [5]:
def build_sine_resistive_frequency_circuit() -> ps.Circuit:
    c = ps.Circuit()
    n_in = c.add_node('in')
    n_out = c.add_node('out')
    gnd = c.ground()
    c.add_sine_voltage_source('Vs', n_in, gnd, 1.0, 1_000.0, 0.0)
    c.add_resistor('R1', n_in, n_out, 1_000.0)
    c.add_resistor('R2', n_out, gnd, 500.0)
    return c


def build_sine_rc_frequency_circuit() -> ps.Circuit:
    c = ps.Circuit()
    n_in = c.add_node('in')
    n_out = c.add_node('out')
    gnd = c.ground()
    c.add_sine_voltage_source('Vs', n_in, gnd, 1.0, 1_000.0, 0.0)
    c.add_resistor('R1', n_in, n_out, 1_000.0)
    c.add_capacitor('C1', n_out, gnd, 1e-6, 0.0)
    return c


# Case A: auto -> periodic
c_periodic = build_sine_resistive_frequency_circuit()
opt_periodic = ps.SimulationOptions()
opt_periodic.dt = 1e-4
opt_periodic.frequency_analysis.enabled = True
opt_periodic.frequency_analysis.mode = ps.FrequencyAnalysisMode.InputImpedance
opt_periodic.frequency_analysis.anchor_mode = ps.FrequencyAnchorMode.Auto
opt_periodic.frequency_analysis.f_start_hz = 10.0
opt_periodic.frequency_analysis.f_stop_hz = 1_000.0
opt_periodic.frequency_analysis.points = 6
opt_periodic.frequency_analysis.injection_current_amplitude = 1.0
opt_periodic.frequency_analysis.perturbation_port.positive_node = 'out'
opt_periodic.periodic_options.max_iterations = 4
opt_periodic.periodic_options.tolerance = 1e-6
opt_periodic.periodic_options.relaxation = 1.0

res_periodic = ps.Simulator(c_periodic, opt_periodic).run_frequency_analysis()

# Case B: auto periodic attempt fails -> averaged fallback
c_fallback = build_sine_rc_frequency_circuit()
opt_fallback = ps.SimulationOptions()
opt_fallback.dt = 1e-3
opt_fallback.periodic_options.period = 7e-4
opt_fallback.periodic_options.max_iterations = 1
opt_fallback.periodic_options.tolerance = 1e-12
opt_fallback.periodic_options.relaxation = 1.0
opt_fallback.frequency_analysis.enabled = True
opt_fallback.frequency_analysis.mode = ps.FrequencyAnalysisMode.InputImpedance
opt_fallback.frequency_analysis.anchor_mode = ps.FrequencyAnchorMode.Auto
opt_fallback.frequency_analysis.sweep_scale = ps.FrequencySweepScale.Linear
opt_fallback.frequency_analysis.f_start_hz = 100.0
opt_fallback.frequency_analysis.f_stop_hz = 1_000.0
opt_fallback.frequency_analysis.points = 4
opt_fallback.frequency_analysis.injection_current_amplitude = 1.0
opt_fallback.frequency_analysis.perturbation_port.positive_node = 'out'

res_fallback = ps.Simulator(c_fallback, opt_fallback).run_frequency_analysis()

print('auto->periodic success:', res_periodic.success, 'anchor:', res_periodic.anchor_mode_selected.name)
print('auto->fallback success:', res_fallback.success, 'anchor:', res_fallback.anchor_mode_selected.name)
print('fallback message:', res_fallback.message)

assert res_periodic.success is True
assert res_periodic.anchor_mode_selected == ps.FrequencyAnchorMode.Periodic
assert res_fallback.success is True
assert res_fallback.anchor_mode_selected == ps.FrequencyAnchorMode.Averaged
assert 'auto fallback applied' in res_fallback.message.lower()

auto->periodic success: True anchor: Periodic
auto->fallback success: True anchor: Averaged
fallback message: Frequency analysis completed (anchor=averaged); auto fallback applied: periodic pre-anchor failed and averaged anchor was selected


## Step 5 - Structured Failure Surface (`FrequencyAnalysisError`)

Demonstrate deterministic failure context for invalid AC setup (zero start frequency).

In [6]:
bad = ps.FrequencyAnalysisOptions()
bad.enabled = True
bad.mode = ps.FrequencyAnalysisMode.OpenLoopTransfer
bad.anchor_mode = ps.FrequencyAnchorMode.DC
bad.f_start_hz = 0.0
bad.f_stop_hz = 1000.0
bad.points = 10
bad.injection_current_amplitude = 1.0
bad.perturbation_port.positive_node = 'in'
bad.output_port.positive_node = 'out'

raw_failure = ps.run_frequency_analysis(circuit, bad)
print('default behavior success:', raw_failure.success)
print('default diagnostic:', raw_failure.diagnostic.name)
print('default message:', raw_failure.message)
assert raw_failure.success is False
assert raw_failure.diagnostic.name == 'FrequencyInvalidConfiguration'

try:
    ps.run_frequency_analysis(circuit, bad, raise_on_failure=True)
    raise AssertionError('Expected FrequencyAnalysisError was not raised')
except ps.FrequencyAnalysisError as err:
    failure_snapshot = {
        'reason_code': err.reason_code,
        'diagnostic': err.diagnostic.name,
        'failed_point_index': err.failed_point_index,
        'failed_frequency_hz_is_nan': math.isnan(err.failed_frequency_hz),
        'mode': err.mode.name,
        'anchor_mode_selected': err.anchor_mode_selected.name,
        'message': err.message,
    }
    print(json.dumps(failure_snapshot, indent=2))
    assert err.reason_code == 'frequency_invalid_configuration'
    assert err.failed_point_index == -1
    assert math.isnan(err.failed_frequency_hz)

default behavior success: False
default diagnostic: FrequencyInvalidConfiguration
default message: Invalid sweep range: require finite f_start_hz > 0 and f_stop_hz >= f_start_hz. [frequency_invalid_configuration]
{
  "reason_code": "frequency_invalid_configuration",
  "diagnostic": "FrequencyInvalidConfiguration",
  "failed_point_index": -1,
  "failed_frequency_hz_is_nan": true,
  "mode": "OpenLoopTransfer",
  "anchor_mode_selected": "Auto",
  "message": "Invalid sweep range: require finite f_start_hz > 0 and f_stop_hz >= f_start_hz. [frequency_invalid_configuration]"
}


## Step 6 - AC Benchmark + Expected-Failure Workflow + KPI Gate

Run the two AC benchmark cases from this spec:

- `ac_rc_lowpass` (analytical AC validation, expected pass),
- `ac_control_workflow_expected_failure` (typed expected failure, counted as pass).

Then freeze a temporary baseline and evaluate `benchmarks/kpi_thresholds_ac.yaml`.

In [7]:
with tempfile.TemporaryDirectory() as td:
    tmp_root = Path(td)
    out_dir = tmp_root / 'bench_out'

    results = benchmark_runner.run_benchmarks(
        benchmarks_path=repo_root / 'benchmarks' / 'benchmarks.yaml',
        output_dir=out_dir,
        selected=['ac_rc_lowpass', 'ac_control_workflow_expected_failure'],
        scenario_filter=['direct_trap'],
    )
    benchmark_runner.write_results(out_dir, results)

    summary = json.loads((out_dir / 'summary.json').read_text(encoding='utf-8'))
    rows = json.loads((out_dir / 'results.json').read_text(encoding='utf-8'))['results']

    print('summary:', summary)
    for row in rows:
        print(
            f"- {row['benchmark_id']}[{row['scenario']}] status={row['status']} mode={row['mode']}"
        )

    rc_row = next(row for row in rows if row['benchmark_id'] == 'ac_rc_lowpass')
    expected_fail_row = next(
        row for row in rows if row['benchmark_id'] == 'ac_control_workflow_expected_failure'
    )

    print('ac_rc_lowpass telemetry keys:', sorted(rc_row['telemetry'].keys()))
    print(
        'ac errors:',
        rc_row['telemetry'].get('ac_sweep_mag_error'),
        rc_row['telemetry'].get('ac_sweep_phase_error'),
    )
    print('expected-failure telemetry:', expected_fail_row['telemetry'])

    assert summary['failed'] == 0
    assert rc_row['status'] == 'passed'
    assert expected_fail_row['status'] == 'passed'
    assert expected_fail_row['telemetry'].get('expected_failure_matched') == 1.0
    assert rc_row['telemetry'].get('ac_sweep_mag_error') is not None
    assert rc_row['telemetry'].get('ac_sweep_phase_error') is not None

    baseline_path, manifest_path = freeze_kpi_baseline.freeze_baseline(
        baseline_id='notebook_ac_demo',
        output_dir=tmp_root / 'baseline',
        bench_results_path=out_dir / 'results.json',
        stress_summary_path=None,
        parity_ltspice_results_path=None,
        parity_ngspice_results_path=None,
        source_artifacts_root=out_dir,
        machine_class_override='notebook',
        cxx_flags_override='-O3',
        overwrite=True,
    )

    gate_report = kpi_gate.run_gate(
        baseline_path=baseline_path,
        thresholds_path=repo_root / 'benchmarks' / 'kpi_thresholds_ac.yaml',
        bench_results_path=out_dir / 'results.json',
        parity_ltspice_results_path=None,
        parity_ngspice_results_path=None,
        stress_summary_path=None,
        strict_provenance=True,
    )

    print('kpi overall_status:', gate_report['overall_status'])
    print('failed_required_metrics:', gate_report['failed_required_metrics'])
    print('ac metric statuses:')
    for key in ('ac_sweep_mag_error', 'ac_sweep_phase_error', 'ac_runtime_p95'):
        item = gate_report['comparisons'].get(key, {})
        print('  ', key, '->', item.get('status'), '|', item.get('message'))

    assert gate_report['overall_status'] == 'passed'
    assert gate_report['failed_required_metrics'] == 0

summary: {'passed': 2, 'failed': 0, 'skipped': 0, 'baseline': 0, 'total': 2}
- ac_rc_lowpass[direct_trap] status=passed mode=frequency_analysis
- ac_control_workflow_expected_failure[direct_trap] status=passed mode=frequency_analysis
ac_rc_lowpass telemetry keys: ['ac_sweep_case', 'ac_sweep_mag_error', 'ac_sweep_mag_rms_error', 'ac_sweep_phase_error', 'ac_sweep_phase_rms_error', 'ac_sweep_points', 'component_declared_count', 'linear_fallbacks', 'linear_iterations', 'linear_solve_calls', 'newton_iterations', 'python_backend', 'residual_norm', 'runtime_kernel_s', 'runtime_s', 'steps', 'timestep_rejections']
ac errors: 7.078021724282735e-09 1.097595259125228e-08
expected-failure telemetry: {'python_backend': 1.0, 'expected_failure_matched': 1.0, 'expected_failure_case': 1.0}


kpi overall_status: passed
failed_required_metrics: 0
ac metric statuses:
   ac_sweep_mag_error -> passed | within threshold
   ac_sweep_phase_error -> passed | within threshold
   ac_runtime_p95 -> passed | within threshold


## Common Pitfalls and Extensions

Common pitfalls:

- running notebook without `build/python` on `PYTHONPATH`,
- forgetting `output_port` in transfer modes,
- treating undefined margins as numeric zeros instead of reading typed reason enums,
- frontend trying to synthesize AC curves instead of plotting backend arrays.

Suggested extensions:

- add one closed-loop AC case once mixed-domain linearization expands,
- add plotting cells (`magnitude_db`, `phase_deg`) with fixed axes for docs screenshots,
- add notebook assertions for new diagnostics as kernel taxonomy evolves.

## Exercises

1. Change RC values (`R`, `C`) in `ac_rc_lowpass.yaml`, rerun Step 2, and verify the cutoff shift.
2. Change sweep from logarithmic to linear and compare phase sampling smoothness.
3. Force a different invalid config (e.g., missing ports) and inspect the diagnostic + reason code.

In [8]:
# Exercise answer scaffold
# Duplicate the objects from previous cells, modify one input at a time,
# rerun ps.run_frequency_analysis(...), and compare summary prints.

# TODO: implement your experiment here.